In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\HP\Desktop\E-Lab\Projects\Kaggle_Ecommerce Data.csv")

In [3]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivered_date'] = pd.to_datetime(df['delivered_date'])
df['delivery_lead_time'] = (
    df['delivered_date'] - df['order_date']
).dt.days

In [4]:
df['is_returned'] = df['returned'].map({'Yes': 1, 'No': 0})

In [5]:
df['return_reason'] = df['return_reason'].fillna('Not Returned')
df.head(5)

,order_id,customer_id,product_id,category,price,discount,quantity,payment_method,order_date,delivered_date,region,returned,return_reason,total_amount,shipping_cost,profit_margin,customer_age,customer_gender,delivery_lead_time,is_returned
0,O100000,C17270,P234890,Home,164.08,0.15,1,Credit Card,2023-12-23,2023-12-27,West,No,Not Returned,139.47,7.88,31.17,60,Female,4,0
1,O100001,C17603,P228204,Grocery,24.73,0.00,1,Credit Card,2025-04-03,2025-04-09,South,No,Not Returned,24.73,4.60,-2.62,37,Male,6,0
2,O100002,C10860,P213892,Electronics,175.58,0.05,1,Credit Card,2024-10-08,2024-10-12,North,No,Not Returned,166.80,6.58,13.44,34,Male,4,0
3,O100003,C15390,P208689,Electronics,63.67,0.00,1,UPI,2024-09-14,2024-09-20,South,No,Not Returned,63.67,5.50,2.14,21,Female,6,0
4,O100004,C15226,P228063,Home,16.33,0.15,1,COD,2024-12-21,2024-12-27,East,No,Not Returned,13.88,2.74,1.15,39,Male,6,0


In [6]:
df['is_returned'].mean() * 100

np.float64(5.515942028985507)

In [7]:
df.groupby('category')['is_returned'].mean().sort_values(ascending=False)

category
Fashion        0.082827
Electronics    0.072977
Home           0.056497
Toys           0.049447
Sports         0.049389
Beauty         0.037777
Grocery        0.013061
Name: is_returned, dtype: float64

In [8]:
df.groupby('region')['is_returned'].mean()

region
Central    0.050959
East       0.059096
North      0.053619
South      0.056830
West       0.054495
Name: is_returned, dtype: float64

In [9]:
df.groupby('payment_method')['is_returned'].mean()

payment_method
COD            0.050962
Credit Card    0.055464
Debit Card     0.056673
PayPal         0.057782
UPI            0.053898
Wallet         0.053753
Name: is_returned, dtype: float64

In [10]:
df[df['is_returned'] == 1]['return_reason'].value_counts()

return_reason
Not as described      490
No longer needed      481
Defective             465
Missing/Wrong item    439
Slow delivery          28
Name: count, dtype: int64

In [11]:
# Price buckets
df['price_bucket'] = pd.cut(
    df['price'],
    bins=[0, 500, 2000, 10000],
    labels=['Low', 'Medium', 'High']
)

In [12]:
# Discount flag
df['high_discount'] = (df['discount'] > 0.2).astype(int)

In [13]:
# COD flag
df['is_cod'] = (df['payment_method'] == 'COD').astype(int)

In [14]:
y = df['is_returned']

In [15]:
features = [
    'price',
    'discount',
    'delivery_lead_time',
    'shipping_cost',
    'profit_margin',
    'is_cod'
]

X = df[features]

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

df['return_probability'] = model.predict_proba(X)[:, 1]

In [20]:
high_risk = df[df['return_probability'] >= threshold]

In [21]:
high_risk.to_csv("high_risk_products.csv", index=False)